# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            This lab uses FREE models only: Ollama (local, no API key needed) and Groq (free tier API). Install Ollama from https://ollama.com or get a free Groq API key from https://console.groq.com
            </span>
        </td>
    </tr>
</table>

In [72]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [73]:
# Using FREE models only: Groq (free tier) and Ollama (local, no API key needed)

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"✓ Groq API Key found (begins {groq_api_key[:4]})")
    print("  Using Groq's FREE tier API")
else:
    print("✗ Groq API Key not set")
    print("  Get a free key at: https://console.groq.com")

# Check if Ollama is available
print("\nChecking Ollama (local, free):")
try:
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=5)
    print("✓ Ollama is installed and available")
    print("\nAvailable models:")
    print(result.stdout)
except FileNotFoundError:
    print("✗ Ollama not found. Please install from https://ollama.com")
except Exception as e:
    print(f"Error checking Ollama: {e}")



✓ Groq API Key found (begins gsk_)
  Using Groq's FREE tier API

Checking Ollama (local, free):
✓ Ollama is installed and available

Available models:
NAME                ID              SIZE      MODIFIED    
deepseek-r1:1.5b    e0979632db5a    1.1 GB    11 days ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    2 weeks ago    
gemma3:1b           8648f39daa8f    815 MB    2 weeks ago    



In [74]:
# Connect to FREE services only

# Groq (free tier API)
groq_url = "https://api.groq.com/openai/v1"
if groq_api_key:
    groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
    print("✓ Connected to Groq API (free tier)")
else:
    groq = None
    print("✗ Groq not available (no API key)")

# Ollama (local, free)
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
print("✓ Connected to Ollama at", ollama_url)


✓ Connected to Groq API (free tier)
✓ Connected to Ollama at http://localhost:11434/v1


In [75]:
# FREE models only!

# Groq models (free tier API - very fast!)
GROQ_LLAMA_70B = "llama-3.3-70b-versatile"
GROQ_LLAMA_8B = "llama-3.1-8b-instant"

# Ollama models (local, free - check what you have installed)
OLLAMA_MODELS = [
    "llama3.2:latest",
    "deepseek-r1:1.5b",
    "gemma3:1b",
]

# Build models list and clients dict
models = []
clients = {}

if groq:
    models.extend([GROQ_LLAMA_70B, GROQ_LLAMA_8B])
    clients[GROQ_LLAMA_70B] = groq
    clients[GROQ_LLAMA_8B] = groq

models.extend(OLLAMA_MODELS)
for model in OLLAMA_MODELS:
    clients[model] = ollama

print(f"\nAvailable FREE models ({len(models)} total):")
for model in models:
    print(f"  - {model}")


Available FREE models (5 total):
  - llama-3.3-70b-versatile
  - llama-3.1-8b-instant
  - llama3.2:latest
  - deepseek-r1:1.5b
  - gemma3:1b


In [76]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (Rev8, Built by MSYS2 project) 15.2.0',
   'g++': 'g++.EXE (Rev8, Built by MSYS2 project) 15.2.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [77]:
# Auto-detect available C++ compiler
import shutil

def find_compiler():
    """Find available C++ compiler on the system"""
    # Try to find g++ (MinGW)
    if shutil.which("g++"):
        return "g++", ["g++", "-std=c++17", "-O3", "-march=native", "-DNDEBUG", "main.cpp", "-o", "main.exe"], ["main.exe"]
    
    # Try to find clang++
    if shutil.which("clang++"):
        return "clang++", ["clang++", "-std=c++17", "-O3", "-march=native", "-DNDEBUG", "main.cpp", "-o", "main.exe"], ["main.exe"]
    
    # Try to find MSVC (cl.exe)
    if shutil.which("cl"):
        return "cl", ["cl", "/O2", "/std:c++17", "/EHsc", "main.cpp"], ["main.exe"]
    
    return None, None, None

compiler_name, compile_command, run_command = find_compiler()

if compiler_name:
    print(f"✓ Found {compiler_name} compiler")
    print(f"  Compile: {' '.join(compile_command)}")
    print(f"  Run: {' '.join(run_command)}")
else:
    print("✗ No C++ compiler found!")
    print("  Install g++ or use online compiler at:")
    print("  https://www.programiz.com/cpp-programming/online-compiler/")

✓ Found g++ compiler
  Compile: g++ -std=c++17 -O3 -march=native -DNDEBUG main.cpp -o main.exe
  Run: main.exe


## And now, on with the main task

In [78]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments with //.
The C++ response needs to produce an identical output in the fastest possible time.
IMPORTANT: If the Python code includes timing, use <chrono> in C++ to measure execution time accurately.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code. Include all necessary headers like <iostream>, <chrono>, and <iomanip>.
Python code to port:

```python
{python}
```
"""

In [79]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [80]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [81]:
def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [82]:
pi = """
import time

# Simple computation with timing
start_time = time.time()

# Do some simple work
result = 0
for i in range(1000000):
    result += i

print(f"Result: {result}")

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution Time: {execution_time:.6f} seconds")
"""

In [83]:
def run_python(code):
    """Simple Python code executor"""
    try:
        exec(code)
    except Exception as e:
        print(f"Error: {e}")

In [84]:
# Run Python version to see baseline performance
print("Python version:")
run_python(pi)

Python version:
Result: 499999500000
Execution Time: 0.101057 seconds


In [85]:
def compile_and_run():
    if compile_command is None:
        print("❌ No C++ compiler available!")
        print("Please install a C++ compiler or use an online compiler:")
        print("https://www.programiz.com/cpp-programming/online-compiler/")
        return
    
    try:
        # Compile
        result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print("✓ Compilation successful!")
        
        # Run 3 times
        for i in range(3):
            result = subprocess.run(run_command, check=True, text=True, capture_output=True)
            print(result.stdout, end='')
    except subprocess.CalledProcessError as e:
        print(f"❌ Error during compilation/execution:")
        print(e.stderr if e.stderr else str(e))
    except FileNotFoundError:
        print("❌ Compiler not found in PATH!")
        print("Please restart VS Code after installing the compiler.")

## Compare Performance: Python vs C++

Now convert the Python code to C++ using the Gradio UI below, then run the C++ version to see the speedup!

In [86]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [88]:
compile_and_run()

✓ Compilation successful!
Result: 499999500000
Execution Time: 0 seconds
Result: 499999500000
Execution Time: 0 seconds
Result: 499999500000
Execution Time: 0 seconds


## Your Experiment Results

**Python Baseline:** ~0.067 seconds

Test the FREE models and record C++ execution times + speedup:

**Groq Models:**
- Llama 3.3 70B: 
  - C++ Time: 0.000022 seconds
  - Speedup: ___ x faster
- Llama 3.1 8B: 
  - C++ Time: ___ seconds
  - Speedup: ___ x faster

**Ollama Local Models:**
- llama3.2:latest: 
  - C++ Time: ___ seconds
  - Speedup: ___ x faster
- deepseek-r1:1.5b: 
  - C++ Time: ___ seconds
  - Speedup: ___ x faster
- gemma3:1b: 
  - C++ Time: ___ seconds
  - Speedup: ___ x faster

**How to calculate speedup:** Divide Python time by C++ time. For example, if Python takes 0.067s and C++ takes 0.001s, the speedup is 67x.

## About This Exercise

This lab focuses on **FREE models only** to make LLM experimentation accessible to everyone:

**Groq (Free Tier API):**
- Fast inference thanks to their custom LPU chips
- Generous free tier for testing
- Get your key at: https://console.groq.com

**Ollama (Local, Always Free):**
- Runs entirely on your machine
- No API keys, no usage limits
- Install from: https://ollama.com

**Why test Python-to-C++ conversion?**
- See how different models handle code generation
- Compare output quality and compilation success
- Learn about performance optimization
- No costs - experiment freely!

**Tip:** For more advanced FREE models, you can also install larger models via Ollama like `qwen2.5-coder:7b` or `deepseek-coder-v2:16b` using `ollama pull <model-name>`

## Code Snippets: How to Print Execution Time

**Python Example:**
```python
import time

start_time = time.time()

# Your code here
result = 0
for i in range(1000000):
    result += i

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution Time: {execution_time:.6f} seconds")
```

**C++ Example:**
```cpp
#include <iostream>
#include <chrono>
#include <iomanip>

int main() {
    auto start_time = std::chrono::high_resolution_clock::now();
    
    // Your code here
    long long result = 0;
    for (int i = 0; i < 1000000; i++) {
        result += i;
    }
    
    auto end_time = std::chrono::high_resolution_clock::now();
    auto duration = std::chrono::duration_cast<std::chrono::duration<double>>(end_time - start_time);
    
    std::cout << "Result: " << result << std::endl;
    std::cout << "Execution Time: " << std::fixed << std::setprecision(6) 
              << duration.count() << " seconds" << std::endl;
    
    return 0;
}
```

**Key Points:**
- Python uses `time.time()` to get timestamps
- C++ uses `<chrono>` library for high-resolution timing
- Both calculate duration by subtracting start from end time
- Format output with 6 decimal places for accuracy